# 02 Pipeline

Build a governed pipeline in six steps: **Environment → Data Contracts → Target → Read → Transform → Write**.

## Tested with FabricOps

This notebook template is maintained separately from FabricOps package releases. The table below records the FabricOps releases that have been manually tested with this template in Microsoft Fabric.

| FabricOps release | Tested by | Date tested |
|---|---|---|
| v0.2.0 | Voyce | 6 Aug 2026 |

This redesigned version has local structural and public-API compatibility validation only. Run it in your configured Fabric workspace before recording a new runtime test.

# 0. Environment

Run the shared Fabric configuration and import the public APIs used by this pipeline.

In [ ]:
%run 00_env_config

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    # FabricOps v0.2.0 onwards
    widget_view_catalogue,
    check_dq,
    check_freshness,
    check_schema,
    check_sensitive_data,
    pipeline_read,
    pipeline_write,
    profile_table,
    resolve_table_id,
    widget_select_data_contract,
)

# 1. Data Contracts

Select the Data Contracts to test with this pipeline. Production automatically uses activated Data Contracts.

In [ ]:
CONTRACTS = widget_select_data_contract()

# 2. Read


In [ ]:
sources = {}

def report_check(name, result):
    """Print the visible outcome of one FabricOps check."""
    status = str((result or {}).get("status", "completed")).upper()
    print(f"  {name}: {status}")

## READ 1 — Orders

In [ ]:
READ_NAME = "orders"
READ_STORE = "source"
READ_SCHEMA = "demo"
READ_TABLE = "orders"
READ_QUERY = None

read_method = "table" if READ_QUERY is None else "query"
print(f"[FabricOps] READ {READ_NAME}")
print(f"  Source: {READ_STORE}.{READ_SCHEMA}.{READ_TABLE}")
print(f"  Method: {read_method} via pipeline_read()")

read_result = pipeline_read(
    store=READ_STORE,
    schema=READ_SCHEMA,
    table_name=READ_TABLE,
    query=READ_QUERY,
)

df = read_result["dataframe"]
table_id = read_result["table_id"]

print(f"  table_id: {table_id}")
print(f"  Data Contract: {'resolved' if read_result['has_contract'] else 'not resolved'}")

if read_result["has_contract"]:
    print("[FabricOps] Freshness")
    print("  Rule source: Data Contract")
    print("  Results: METADATA_GUARDRAIL_RESULTS")
    freshness_result = check_freshness(table_id, raise_on_failure=True)
    report_check("Freshness", freshness_result)

    print("[FabricOps] Schema")
    print("  Rule source: Data Contract")
    print("  Results: METADATA_GUARDRAIL_RESULTS")
    schema_result = check_schema(table_id, dataframe=df, raise_on_failure=True)
    report_check("Schema", schema_result)

    print("[FabricOps] Data Quality")
    print("  Rule source: Data Contract")
    print("  Results: METADATA_GUARDRAIL_RESULTS")
    print("  Row failures: METADATA_GUARDRAIL_ROW_RESULTS when applicable")
    dq_result = check_dq(df, table_id=table_id, raise_on_failure=True)
    report_check("Data Quality", dq_result)
else:
    print("[FabricOps] Governed checks skipped")
    print("  No Data Contract resolved for this table_id.")
    print("  Skipped: Freshness, Source Stability, Schema, Data Quality")

print("[FabricOps] Profile")
print(f"  Source: {READ_STORE}.{READ_SCHEMA}.{READ_TABLE}")
print("  Method: profile_table(table_id=...)")
print("  Writes: METADATA_DATA_CATALOGUE")
print("          METADATA_DATA_PROFILED")
print("          METADATA_DATA_PROFILED_FREQUENCY where applicable")
profile = profile_table(table_id=table_id)
display(profile["profile"])

sources[READ_NAME] = {
    "dataframe": df,
    "table_id": table_id,
    "has_contract": read_result["has_contract"],
}

# Optional: inspect this source in the current pipeline catalogue.
# catalogue_widget = widget_view_catalogue(mode="pipeline")
# catalogue_widget["show"](table_id=table_id)

## READ 2 — Products

In [ ]:
READ_NAME = "products"
READ_STORE = "source"
READ_SCHEMA = "demo"
READ_TABLE = "products"
READ_QUERY = None

read_method = "table" if READ_QUERY is None else "query"
print(f"[FabricOps] READ {READ_NAME}")
print(f"  Source: {READ_STORE}.{READ_SCHEMA}.{READ_TABLE}")
print(f"  Method: {read_method} via pipeline_read()")

read_result = pipeline_read(
    store=READ_STORE,
    schema=READ_SCHEMA,
    table_name=READ_TABLE,
    query=READ_QUERY,
)

df = read_result["dataframe"]
table_id = read_result["table_id"]

print(f"  table_id: {table_id}")
print(f"  Data Contract: {'resolved' if read_result['has_contract'] else 'not resolved'}")

if read_result["has_contract"]:
    print("[FabricOps] Freshness")
    print("  Rule source: Data Contract")
    print("  Results: METADATA_GUARDRAIL_RESULTS")
    freshness_result = check_freshness(table_id, raise_on_failure=True)
    report_check("Freshness", freshness_result)

    print("[FabricOps] Schema")
    print("  Rule source: Data Contract")
    print("  Results: METADATA_GUARDRAIL_RESULTS")
    schema_result = check_schema(table_id, dataframe=df, raise_on_failure=True)
    report_check("Schema", schema_result)

    print("[FabricOps] Data Quality")
    print("  Rule source: Data Contract")
    print("  Results: METADATA_GUARDRAIL_RESULTS")
    print("  Row failures: METADATA_GUARDRAIL_ROW_RESULTS when applicable")
    dq_result = check_dq(df, table_id=table_id, raise_on_failure=True)
    report_check("Data Quality", dq_result)
else:
    print("[FabricOps] Governed checks skipped")
    print("  No Data Contract resolved for this table_id.")
    print("  Skipped: Freshness, Source Stability, Schema, Data Quality")

print("[FabricOps] Profile")
print(f"  Source: {READ_STORE}.{READ_SCHEMA}.{READ_TABLE}")
print("  Method: profile_table(table_id=...)")
print("  Writes: METADATA_DATA_CATALOGUE")
print("          METADATA_DATA_PROFILED")
print("          METADATA_DATA_PROFILED_FREQUENCY where applicable")
profile = profile_table(table_id=table_id)
display(profile["profile"])

sources[READ_NAME] = {
    "dataframe": df,
    "table_id": table_id,
    "has_contract": read_result["has_contract"],
}

# Optional: inspect this source in the current pipeline catalogue.
# catalogue_widget = widget_view_catalogue(mode="pipeline")
# catalogue_widget["show"](table_id=table_id)

## READ 3 — Order History

In [ ]:
READ_NAME = "history"
READ_STORE = "product"
READ_SCHEMA = "demo"
READ_TABLE = "order_history"
READ_QUERY = None

read_method = "table" if READ_QUERY is None else "query"
print(f"[FabricOps] READ {READ_NAME}")
print(f"  Source: {READ_STORE}.{READ_SCHEMA}.{READ_TABLE}")
print(f"  Method: {read_method} via pipeline_read()")

read_result = pipeline_read(
    store=READ_STORE,
    schema=READ_SCHEMA,
    table_name=READ_TABLE,
    query=READ_QUERY,
)

df = read_result["dataframe"]
table_id = read_result["table_id"]

print(f"  table_id: {table_id}")
print(f"  Data Contract: {'resolved' if read_result['has_contract'] else 'not resolved'}")

if read_result["has_contract"]:
    print("[FabricOps] Freshness")
    print("  Rule source: Data Contract")
    print("  Results: METADATA_GUARDRAIL_RESULTS")
    freshness_result = check_freshness(table_id, raise_on_failure=True)
    report_check("Freshness", freshness_result)

    print("[FabricOps] Schema")
    print("  Rule source: Data Contract")
    print("  Results: METADATA_GUARDRAIL_RESULTS")
    schema_result = check_schema(table_id, dataframe=df, raise_on_failure=True)
    report_check("Schema", schema_result)

    print("[FabricOps] Data Quality")
    print("  Rule source: Data Contract")
    print("  Results: METADATA_GUARDRAIL_RESULTS")
    print("  Row failures: METADATA_GUARDRAIL_ROW_RESULTS when applicable")
    dq_result = check_dq(df, table_id=table_id, raise_on_failure=True)
    report_check("Data Quality", dq_result)
else:
    print("[FabricOps] Governed checks skipped")
    print("  No Data Contract resolved for this table_id.")
    print("  Skipped: Freshness, Source Stability, Schema, Data Quality")

print("[FabricOps] Profile")
print(f"  Source: {READ_STORE}.{READ_SCHEMA}.{READ_TABLE}")
print("  Method: profile_table(table_id=...)")
print("  Writes: METADATA_DATA_CATALOGUE")
print("          METADATA_DATA_PROFILED")
print("          METADATA_DATA_PROFILED_FREQUENCY where applicable")
profile = profile_table(table_id=table_id)
display(profile["profile"])

sources[READ_NAME] = {
    "dataframe": df,
    "table_id": table_id,
    "has_contract": read_result["has_contract"],
}

# Optional: inspect this source in the current pipeline catalogue.
# catalogue_widget = widget_view_catalogue(mode="pipeline")
# catalogue_widget["show"](table_id=table_id)

# 3. Transform


In [ ]:
orders_df = sources["orders"]["dataframe"]
products_df = sources["products"]["dataframe"]
history_df = sources["history"]["dataframe"]

history_summary_df = (
    history_df
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("historical_order_count"),
        F.sum("net_amount").alias("historical_net_amount"),
        F.max("order_datetime").alias("latest_historical_order_datetime"),
    )
)

transformed_df = (
    orders_df.alias("orders")
    .join(products_df.alias("products"), on="product_id", how="left")
    .join(history_summary_df.alias("history"), on="customer_id", how="left")
    .withColumn(
        "order_net_amount",
        F.round(F.col("quantity") * F.col("unit_price") * (F.lit(1.0) - F.col("discount")), 2),
    )
    .fillna({"historical_order_count": 0, "historical_net_amount": 0.0})
    .select(
        "order_id", "customer_id", "order_datetime", "modified_datetime",
        "product_id", "product_name", "product_category", "quantity", "unit_price",
        "discount", "order_net_amount", "order_status", "shipping_country",
        "historical_order_count", "historical_net_amount", "latest_historical_order_datetime",
    )
)

display(transformed_df)

# 4. Target


In [ ]:
WRITE_STORE = "unified"
WRITE_SCHEMA = "demo"
WRITE_TABLE = "curated_orders"
WRITE_LOAD_STRATEGY = "overwrite"

WRITE_TABLE_ID = resolve_table_id(
    store=WRITE_STORE,
    schema=WRITE_SCHEMA,
    table_name=WRITE_TABLE,
)

print("[FabricOps] TARGET")
print(f"  Destination: {WRITE_STORE}.{WRITE_SCHEMA}.{WRITE_TABLE}")
print(f"  table_id: {WRITE_TABLE_ID}")
print(f"  Write method: pipeline_write(load_strategy={WRITE_LOAD_STRATEGY!r})")

# 5. Write


## WRITE 1 — Curated Orders

In [ ]:
print("[FabricOps] WRITE")
print(f"  Destination: {WRITE_STORE}.{WRITE_SCHEMA}.{WRITE_TABLE}")
print(f"  table_id: {WRITE_TABLE_ID}")
print(f"  Method: pipeline_write(load_strategy={WRITE_LOAD_STRATEGY!r})")
print("  Guardrail results: METADATA_GUARDRAIL_RESULTS")
print("  Row failures: METADATA_GUARDRAIL_ROW_RESULTS when applicable")

print("[FabricOps] Target Schema")
print("  Rule source: Data Contract")
schema_result = check_schema(WRITE_TABLE_ID, dataframe=transformed_df, raise_on_failure=True)
report_check("Target Schema", schema_result)

print("[FabricOps] Target Data Quality")
print("  Rule source: Data Contract")
dq_result = check_dq(transformed_df, table_id=WRITE_TABLE_ID, raise_on_failure=True)
report_check("Target Data Quality", dq_result)

print("[FabricOps] Sensitive Data")
print("  Rule source: Data Contract")
print("  Results: METADATA_GUARDRAIL_RESULTS")
sensitive_result = check_sensitive_data(
    transformed_df,
    table_id=WRITE_TABLE_ID,
)
report_check("Sensitive Data", sensitive_result)
if not sensitive_result["can_continue"]:
    raise RuntimeError("A blocking Sensitive Data Guardrail failed for the target.")

prepared_df = sensitive_result["dataframe"]

print("[FabricOps] Publish")
print(f"  Writing data to: {WRITE_STORE}.{WRITE_SCHEMA}.{WRITE_TABLE}")
print(f"  Load strategy: {WRITE_LOAD_STRATEGY}")
print("  On successful write:")
print("    Lineage -> METADATA_DATA_LINEAGE")
print("    Source observation state -> METADATA_SOURCE_OBSERVATION")
write_result = pipeline_write(
    prepared_df,
    store=WRITE_STORE,
    schema=WRITE_SCHEMA,
    table_name=WRITE_TABLE,
    load_strategy=WRITE_LOAD_STRATEGY,
    source_table_ids=[source["table_id"] for source in sources.values()],
)

print("[FabricOps] Target Profile")
print(f"  Profiling persisted target table_id: {write_result['table_id']}")
print("  Writes: METADATA_DATA_CATALOGUE")
print("          METADATA_DATA_PROFILED")
print("          METADATA_DATA_PROFILED_FREQUENCY where applicable")
write_profile = profile_table(table_id=write_result["table_id"])
display(write_profile["profile"])

# Optional: inspect the persisted target in the current pipeline catalogue.
# catalogue_widget = widget_view_catalogue(mode="pipeline")
# catalogue_widget["show"](table_id=write_result["table_id"])